In [76]:
import datetime

import pandas as pd

from beancount.loader import load_string

from beancount.parser.printer import print_errors

from evbeantools.juptools import get_net_worths, get_bean_pivot
from evbeantools.printer_rich import display_entries

In [77]:
ledger = """

2020-01-01 open Equity:Opening-Balance
2020-01-01 open Assets:Cash
2020-01-01 open Expenses:Misc
2020-01-01 open Liabilities:Credit-Card
2020-01-01 open Income:VacH 
2020-01-01 open Assets:VacH

2020-01-01 * "Initial balance"
    Assets:Cash          1000.00 USD
    Equity:Opening-Balance

2020-12-01 * "Buy groceries Cash"
    Assets:Cash           -100.00 USD
    Expenses:Misc
    

    
2020-12-01 * "Buy groceries CC"
    Assets:Cash           -100.00 USD
    Liabilities:Credit-Card
    
2021-12-01 * "Buy groceries"
    Assets:Cash           -100.00 USD
    Expenses:Misc
    
"""

entries, errors, options = load_string(ledger)

if errors:
    print_errors(errors)




In [78]:
net_worths_dates = [datetime.date(2020, 12, 31), datetime.date(2021, 12, 31), datetime.date(2022, 12, 31)]

net_worths = get_net_worths(entries, options, net_worths_dates, 'USD')

net_worths

acc_L0       acc_L1 amount (USD)                      
q_date                             2020-12-31 2021-12-31 2022-12-31
0            Assets         Cash        800.0      700.0      700.0
1       Liabilities  Credit-Card        100.0      100.0      100.0

In [79]:
net_worths.columns

MultiIndex([(      'acc_L0',         ''),
            (      'acc_L1',         ''),
            ('amount (USD)', 2020-12-31),
            ('amount (USD)', 2021-12-31),
            ('amount (USD)', 2022-12-31)],
           names=[None, 'q_date'])

In [80]:
net_worths.index

RangeIndex(start=0, stop=2, step=1)

In [81]:
test_df_data = {
    "amount (EUR)": [1000.0, 100.0, -100.0],
    "account": ["Expenses:Misc:Misc1", "Expenses:Food", "Expenses:Entertainment"],
    "year": [2020, 2020, 2021]
}

test_df = pd.DataFrame(test_df_data)
test_df

,amount (EUR),account,year
0,1000.0,Expenses:Misc:Misc1,2020
1,100.0,Expenses:Food,2020
2,-100.0,Expenses:Entertainment,2021


In [82]:
pivot_df = get_bean_pivot(test_df,  max_row_levels=10, repeat_row_labels=False, drop_identical_row_levels=False)
pivot_df

amount (EUR)       
year                                  2020   2021
acc_L0   acc_L1        acc_L2                    
Expenses Entertainment _               0.0 -100.0
         Food          _             100.0    0.0
         Misc          Misc1        1000.0    0.0

In [83]:
pivot_df.index

MultiIndex([('Expenses', 'Entertainment',     '_'),
            ('Expenses',          'Food',     '_'),
            ('Expenses',          'Misc', 'Misc1')],
           names=['acc_L0', 'acc_L1', 'acc_L2'])

In [84]:
pivot_df.columns

MultiIndex([('amount (EUR)', 2020),
            ('amount (EUR)', 2021)],
           names=[None, 'year'])

In [89]:
pivot_df.style.set_table_styles([{'selector': 'th', 'props': [('rowspan', '1'), ('colspan', '1')]}])